# **Proyek ETL Pipeline Bank Marketing (Databricks Free Edition)**
## **Layer 3: Gold Layer — Business Data Marts (Task 3 Workflow)**

Notebook ini berperan sebagai **Task 3 (`Aggregate_Gold`)** yang menunggu **Task 2 (`Clean_Silver`)** selesai.

### **Desain Tabel Delta Gold & Grain Eksplisit (Sumber AI/BI Dashboard)**

| Tabel Delta Gold | Grain (1 Baris Mewakili) | Dimensi Utama | Metrik Utama | Menjawab Kebutuhan |
|---|---|---|---|---|
| `gold_monthly_campaign_trend` | 1 bulan kampanye (`year_month`) | `campaign_year`, `month_num`, `month`, `year_month`, `campaign_date` | `total_klien`, `total_subscribed`, `conversion_rate_pct`, `avg_balance`, `avg_duration_sec` | **Pertanyaan #3 (Tren Waktu)** & **KPI Ringkas Dashboard** |
| `gold_education_marital_summary` | 1 kombinasi `education` × `marital` | `education`, `marital` | `total_klien`, `total_subscribed`, `conversion_rate_pct`, `avg_balance`, `avg_duration_sec` | **Pertanyaan #1** & **Perbandingan Kategori Dashboard** |
| `gold_age_balance_summary` | 1 kombinasi `age_group` × `balance_segment` | `age_group`, `balance_segment` | `total_klien`, `total_subscribed`, `conversion_rate_pct`, `avg_balance`, `min_balance`, `max_balance` | **Pertanyaan #2 & #3** & **Perbandingan Segmen Dashboard** |

---
### **1. Parameter Task Workflow (`dbutils.widgets`)**

In [ ]:
import json
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace", "Unity Catalog Name")
dbutils.widgets.text("schema", "bank_marketing", "Schema Name")
dbutils.widgets.text("silver_table", "silver_bank_marketing", "Nama Tabel Delta Silver")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
silver_table = dbutils.widgets.get("silver_table").strip()

full_silver_table = f"{catalog}.{schema}.{silver_table}"
gold_trend_table = f"{catalog}.{schema}.gold_monthly_campaign_trend"
gold_edu_marital_table = f"{catalog}.{schema}.gold_education_marital_summary"
gold_age_balance_table = f"{catalog}.{schema}.gold_age_balance_summary"

if not spark.catalog.tableExists(full_silver_table):
    raise ValueError(
        f"Tabel Silver {full_silver_table} belum tersedia. "
        "Pastikan Task 2 (02_Silver_Layer) berhasil dijalankan terlebih dahulu."
    )

df_silver = spark.table(full_silver_table)
silver_count = df_silver.count()
print(f"Sumber Silver: {full_silver_table} ({silver_count:,} baris)")

---
### **2. Pembentukan 3 Tabel Agregasi Gold (Delta)**

In [ ]:
def safe_conversion_rate():
    return F.when(
        F.count("*") > 0,
        F.round(F.sum("y_flag") * 100.0 / F.count("*"), 2),
    ).otherwise(F.lit(0.0))


# ------------------------------------------------------------------------------
# Tabel Gold 1: Tren Kampanye Bulanan (Grain: 1 baris = 1 bulan / year_month)
# ------------------------------------------------------------------------------
df_gold_trend = (
    df_silver
    .groupBy("campaign_year", "month_num", "month", "year_month", "campaign_date")
    .agg(
        F.count("*").alias("total_klien"),
        F.sum("y_flag").alias("total_subscribed"),
        F.sum(1 - F.col("y_flag")).alias("total_not_subscribed"),
        safe_conversion_rate().alias("conversion_rate_pct"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.min("balance").alias("min_balance"),
        F.max("balance").alias("max_balance"),
        F.round(F.avg("duration"), 2).alias("avg_duration_sec"),
        F.round(F.avg("campaign"), 2).alias("avg_campaign_contacts"),
    )
    .withColumn("gold_updated_at", F.current_timestamp())
    .orderBy("campaign_date")
)

(
    df_gold_trend.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_trend_table)
)
spark.sql(
    f"COMMENT ON TABLE {gold_trend_table} IS "
    "'Gold Layer (Grain: year_month): tren bulanan volume kontak, konversi deposito, saldo, dan durasi panggilan (Mei 2008 - Nov 2010).'"
)

# ------------------------------------------------------------------------------
# Tabel Gold 2: Ringkasan Pendidikan x Status Pernikahan
# ------------------------------------------------------------------------------
df_gold_edu_marital = (
    df_silver
    .groupBy("education", "marital")
    .agg(
        F.count("*").alias("total_klien"),
        F.sum("y_flag").alias("total_subscribed"),
        F.sum(1 - F.col("y_flag")).alias("total_not_subscribed"),
        safe_conversion_rate().alias("conversion_rate_pct"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.round(F.avg("duration"), 2).alias("avg_duration_sec"),
    )
    .withColumn("gold_updated_at", F.current_timestamp())
    .orderBy(F.col("conversion_rate_pct").desc(), F.col("total_klien").desc())
)

(
    df_gold_edu_marital.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_edu_marital_table)
)
spark.sql(
    f"COMMENT ON TABLE {gold_edu_marital_table} IS "
    "'Gold Layer (Grain: education x marital): perbandingan tingkat konversi deposito dan saldo rata-rata.'"
)

# ------------------------------------------------------------------------------
# Tabel Gold 3: Ringkasan Kelompok Usia x Segmen Saldo
# ------------------------------------------------------------------------------
age_order_expr = (
    F.when(F.col("age_group") == "Muda (<30)", 1)
    .when(F.col("age_group") == "Paruh Baya (30-50)", 2)
    .otherwise(3)
)
bal_order_expr = (
    F.when(F.col("balance_segment") == "Negatif (<0)", 1)
    .when(F.col("balance_segment") == "Rendah (0-500)", 2)
    .when(F.col("balance_segment") == "Menengah (501-2000)", 3)
    .otherwise(4)
)

df_gold_age_balance = (
    df_silver
    .groupBy("age_group", "balance_segment")
    .agg(
        F.count("*").alias("total_klien"),
        F.sum("y_flag").alias("total_subscribed"),
        F.sum(1 - F.col("y_flag")).alias("total_not_subscribed"),
        safe_conversion_rate().alias("conversion_rate_pct"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.min("balance").alias("min_balance"),
        F.max("balance").alias("max_balance"),
        F.round(F.avg("duration"), 2).alias("avg_duration_sec"),
    )
    .withColumn("age_group_order", age_order_expr)
    .withColumn("balance_segment_order", bal_order_expr)
    .withColumn("gold_updated_at", F.current_timestamp())
    .orderBy("age_group_order", "balance_segment_order")
)

(
    df_gold_age_balance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_age_balance_table)
)
spark.sql(
    f"COMMENT ON TABLE {gold_age_balance_table} IS "
    "'Gold Layer (Grain: age_group x balance_segment): perbandingan konversi deposito menurut kelompok usia dan segmen saldo.'"
)

print("Ketiga tabel Delta Gold berhasil diperbarui secara idempotent.")

---
### **3. Verifikasi Tabel Delta Gold & Ringkasan KPI Global**

In [ ]:
trend_df = spark.table(gold_trend_table)
edu_df = spark.table(gold_edu_marital_table)
age_bal_df = spark.table(gold_age_balance_table)

print(f"1) {gold_trend_table} ({trend_df.count()} baris / bulan)")
display(trend_df.orderBy("campaign_date"))

print(f"2) {gold_edu_marital_table} ({edu_df.count()} kombinasi)")
display(edu_df)

print(f"3) {gold_age_balance_table} ({age_bal_df.count()} kombinasi)")
display(age_bal_df)

kpi_row = trend_df.agg(
    F.sum("total_klien").alias("total_klien"),
    F.sum("total_subscribed").alias("total_subscribed"),
    F.round(F.sum("total_subscribed") * 100.0 / F.sum("total_klien"), 2).alias("overall_conversion_rate_pct"),
).collect()[0]

print("=== KPI RINGKAS GLOBAL (DARI GOLD) ===")
print(f"Total Klien               : {kpi_row['total_klien']:,}")
print(f"Total Berlangganan (yes)  : {kpi_row['total_subscribed']:,}")
print(f"Conversion Rate Keseluruhan: {kpi_row['overall_conversion_rate_pct']}%")

result_payload = {
    "status": "SUCCESS",
    "tables": {
        "trend": gold_trend_table,
        "education_marital": gold_edu_marital_table,
        "age_balance": gold_age_balance_table,
    },
    "total_klien": int(kpi_row["total_klien"]),
    "total_subscribed": int(kpi_row["total_subscribed"]),
    "overall_conversion_rate_pct": float(kpi_row["overall_conversion_rate_pct"]),
}
dbutils.notebook.exit(json.dumps(result_payload))